# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library. We will perform a step-by-step exploration using the Croissant metadata schema, referencing all entities by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets and their IDs, fields, and field IDs as provided by the Croissant schema.

We'll list all record sets, and for each record set, list its available fields (columns) and their `@id`s.

In [ ]:
# List all record sets
record_set_objs = dataset.metadata.record_sets
record_set_ids = []

print("Available Record Sets:\n----------------------")
for rs in record_set_objs:
    print(f"Record Set: {rs.name} [@id: {rs.id_}]")
    record_set_ids.append(rs.id_)
    print("  Fields (columns):")
    for field in rs.fields:
        print(f"    {field.name} [@id: {field.id_}]")
    print()

# If only one main record set is present, print its details
if len(record_set_ids) == 1:
    main_record_set_id = record_set_ids[0]
else:
    main_record_set_id = record_set_ids[0]  # Default to first one

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set and store in a dictionary of DataFrames (by record set @id)
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"  {len(records)} records loaded.")
        print(f"  Columns: {dataframes[rs_id].columns.tolist()}")
        print()
    except Exception as e:
        print(f"  Could not load records: {str(e)}\n")

# Display columns and first few rows of the main record set
if main_record_set_id in dataframes:
    print(f"Columns in main record set (@id: {main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply common data processing steps, such as filtering records based on specific criteria, normalizing a numeric field, and grouping by a categorical field. We refer to all fields by their unique `@id`.

Below, substitute the correct `@id`s as needed based on your record set and field list from above.

In [ ]:
# Pick one numeric field and one group/categorical field for analysis by their @id
# You can adjust these according to metadata from previous cells

# Example: suppose the main field ids in this dataset are as follows (adjust accordingly):
numeric_field_id = None
group_field_id = None
# Infer field IDs from the DataFrame columns if possible
df = dataframes.get(main_record_set_id)
if df is not None:
    # Try to pick a numeric field by name heuristic
    for col in df.columns:
        if any(s in col.lower() for s in ['abundance', 'count', 'coverage', 'mw', 'pi', 'score', 'peptide']):
            numeric_field_id = col
            break
    for col in df.columns:
        if any(s in col.lower() for s in ['group', 'sample', 'type', 'annotation', 'protein', 'accession']):
            group_field_id = col
            break

if numeric_field_id is None:
    numeric_field_id = df.columns[0]  # fallback
if group_field_id is None:
    group_field_id = df.columns[1]  # fallback

print(f"Using numeric field (by @id): {numeric_field_id}")
print(f"Using group field (by @id): {group_field_id}")

# Attempt to convert numeric field to numeric type
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter to rows with numeric_field_id > threshold (e.g., mean value)
threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field and show mean statistics
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    grouped_df = grouped_df.rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we provide an example histogram and bar chart using the extracted numeric/group fields. You can adapt the visualizations as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Bar plot of grouped means
if 'grouped_df' in locals() and len(grouped_df) > 0:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=grouped_df, x=group_field_id, y=f"mean_{numeric_field_id}")
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, explore, and process a Croissant-based scientific dataset using the `mlcroissant` library. We listed available record sets and their fields by their unique `@id`, extracted tabular data from the record set, filtered and normalized numeric features, and visualized field distributions. This workflow can be extended for more advanced analyses, such as feature engineering or machine learning, using the FAIR^2 dataset schema and data resources.